# Training Sentiment Analysis - Bank Jago

Melatih 6 model eksperimen (EXP-01 s/d EXP-06) untuk klasifikasi sentimen (Positive/Neutral/Negative).
Hasil: model artifacts (`models/exp-*/`) + classification reports (`reports/`) + `experiment_results.json`.

In [1]:
import os, json, joblib, re, warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from datetime import datetime

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
SENTIMENT_LABELS = ['Negative', 'Neutral', 'Positive']

os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

## 1. Load & Label Data

In [2]:
def rating_to_label(score):
    if score <= 2: return 0
    if score == 3: return 1
    return 2

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('../data/raw/reviews.csv')
df['label'] = df['score'].apply(rating_to_label)
df['clean'] = df['content'].apply(clean_text)
df = df[df['clean'].str.len() > 0].reset_index(drop=True)

print(f'Total samples: {len(df)}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().sort_index())
print(f'  Negative: {(df["label"]==0).sum()}, Neutral: {(df["label"]==1).sum()}, Positive: {(df["label"]==2).sum()}')

Total samples: 9916

Label distribution:
label
0    2649
1     315
2    6952
Name: count, dtype: int64
  Negative: 2649, Neutral: 315, Positive: 6952


## 2. Train/Test Split

In [3]:
X = np.array(df['clean'].values)
y = np.array(df['label'].values).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED, shuffle=True
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 7932, Test: 1984


## 3. Experiment 1-3: TF-IDF / BoW + LR / SVM

In [4]:
experiments = [
    {
        'id': 'EXP-01_LR_TFIDF',
        'model_name': 'LogisticRegression',
        'vectorizer_params': {'max_features': 10000, 'ngram_range': (1, 2)},
        'model_class': LogisticRegression,
        'model_args': {'max_iter': 1000, 'class_weight': 'balanced', 'random_state': SEED, 'n_jobs': -1},
    },
    {
        'id': 'EXP-02_SVM_TFIDF',
        'model_name': 'LinearSVC',
        'vectorizer_params': {'max_features': 15000, 'ngram_range': (1, 2)},
        'model_class': LinearSVC,
        'model_args': {'max_iter': 2000, 'class_weight': 'balanced', 'random_state': SEED},
    },
    {
        'id': 'EXP-03_LR_BoW',
        'model_name': 'LogisticRegression_BoW',
        'vectorizer_params': {'max_features': 8000, 'ngram_range': (1, 1)},
        'model_class': LogisticRegression,
        'model_args': {'max_iter': 1000, 'class_weight': 'balanced', 'random_state': SEED, 'n_jobs': -1},
    },
]

In [5]:
results_list = []

for exp in experiments:
    print(f'\n{"="*60}')
    print(f"Running {exp['id']} ({exp['model_name']})")
    print('='*60)

    # Vectorize
    vectorizer = TfidfVectorizer(**exp['vectorizer_params'], stop_words=None, sublinear_tf=True)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    # Train
    model = exp['model_class'](**exp['model_args'])
    model.fit(X_train_vec, y_train)

    # Predict & Metrics
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred).tolist()

    # Per-class
    p_per, r_per, f1_per, s_per = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1, 2], zero_division=0)
    per_class = {}
    for i, label in enumerate(SENTIMENT_LABELS):
        per_class[label] = {
            'precision': round(p_per[i], 4),
            'recall': round(r_per[i], 4),
            'f1': round(f1_per[i], 4),
            'support': int(s_per[i]),
        }

    cls_report = classification_report(y_test, y_pred, target_names=SENTIMENT_LABELS, zero_division=0)
    print(cls_report)
    print(f'Accuracy: {acc:.2%} | F1 (macro): {f1:.4f}')

    # Save artifacts
    model_dir = f"../models/{exp['id']}"
    os.makedirs(model_dir, exist_ok=True)
    joblib.dump(model, f'{model_dir}/model.pkl')
    joblib.dump(vectorizer, f'{model_dir}/vectorizer.pkl')
    print(f'Model saved to {model_dir}/')

    # Save report
    with open(f"../reports/{exp['id']}_classification.txt", 'w') as f:
        f.write(cls_report)

    # Store result
    result = {
        'experiment': exp['id'],
        'model': exp['model_name'],
        'vectorizer_params': exp['vectorizer_params'],
        'split': {'train': len(X_train), 'test': len(X_test), 'ratio': 0.2, 'stratified': True},
        'metrics': {
            'accuracy': round(acc, 4),
            'precision_macro': round(precision, 4),
            'recall_macro': round(recall, 4),
            'f1_macro': round(f1, 4),
            'per_class': per_class,
            'confusion_matrix': cm,
        },
        'timestamp': datetime.now().isoformat(),
        'status': 'completed',
    }
    results_list.append(result)


Running EXP-01_LR_TFIDF (LogisticRegression)


              precision    recall  f1-score   support

    Negative       0.78      0.89      0.83       530
     Neutral       0.18      0.22      0.20        63
    Positive       0.97      0.91      0.94      1391

    accuracy                           0.88      1984
   macro avg       0.65      0.68      0.66      1984
weighted avg       0.90      0.88      0.89      1984

Accuracy: 88.46% | F1 (macro): 0.6581


Model saved to ../models/EXP-01_LR_TFIDF/

Running EXP-02_SVM_TFIDF (LinearSVC)


              precision    recall  f1-score   support

    Negative       0.80      0.89      0.84       530
     Neutral       0.19      0.08      0.11        63
    Positive       0.95      0.94      0.94      1391

    accuracy                           0.90      1984
   macro avg       0.65      0.63      0.63      1984
weighted avg       0.89      0.90      0.89      1984

Accuracy: 89.57% | F1 (macro): 0.6321


Model saved to ../models/EXP-02_SVM_TFIDF/

Running EXP-03_LR_BoW (LogisticRegression_BoW)


              precision    recall  f1-score   support

    Negative       0.79      0.88      0.83       530
     Neutral       0.15      0.24      0.19        63
    Positive       0.97      0.91      0.94      1391

    accuracy                           0.88      1984
   macro avg       0.64      0.68      0.65      1984
weighted avg       0.90      0.88      0.89      1984

Accuracy: 88.00% | F1 (macro): 0.6539
Model saved to ../models/EXP-03_LR_BoW/


## 4. Experiment 4: Ensemble LR + SVM (Soft Voting)

In [6]:
from sklearn.calibration import CalibratedClassifierCV

print(f'\n{"="*60}')
print('Running EXP-04_Ensemble (LR + SVM Soft Voting)')
print('='*60)

# Extended TF-IDF vectorizer (matching script: 1-3 gram)
ensemble_vec = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), stop_words=None, sublinear_tf=True)
X_train_ens = ensemble_vec.fit_transform(X_train)
X_test_ens = ensemble_vec.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1)
lr.fit(X_train_ens, y_train)

# Calibrate SVM for probability output
svm = CalibratedClassifierCV(LinearSVC(max_iter=2000, class_weight='balanced', random_state=SEED))
svm.fit(X_train_ens, y_train)

# Soft voting
lr_proba = lr.predict_proba(X_test_ens)
svm_proba = svm.predict_proba(X_test_ens)
ensemble_proba = (lr_proba + svm_proba) / 2
y_pred_ens = np.argmax(ensemble_proba, axis=1)

acc_ens = accuracy_score(y_test, y_pred_ens)
p_per_e, r_per_e, f1_per_e, s_per_e = precision_recall_fscore_support(y_test, y_pred_ens, labels=[0, 1, 2], zero_division=0)
cm_ens = confusion_matrix(y_test, y_pred_ens).tolist()
cls_ens = classification_report(y_test, y_pred_ens, target_names=SENTIMENT_LABELS, zero_division=0)

print(f'Accuracy: {acc_ens:.2%}')
print(cls_ens)

# Save ensemble
ens_dir = '../models/EXP-04_Ensemble'
os.makedirs(ens_dir, exist_ok=True)
joblib.dump({'lr': lr, 'svm': svm}, f'{ens_dir}/model.pkl')
joblib.dump(ensemble_vec, f'{ens_dir}/vectorizer.pkl')
with open(f'../reports/EXP-04_Ensemble_classification.txt', 'w') as f:
    f.write(cls_ens)
print(f'Ensemble model saved to {ens_dir}/')

ens_result = {
    'experiment': 'EXP-04_Ensemble',
    'model': 'Ensemble_LR_SVM',
    'vectorizer_params': {'max_features': 15000, 'ngram_range': [1, 3]},
    'split': {'train': len(X_train), 'test': len(X_test), 'ratio': 0.2, 'stratified': True},
    'metrics': {
        'accuracy': round(acc_ens, 4),
        'per_class': {label: {'precision': round(p_per_e[i], 4), 'recall': round(r_per_e[i], 4), 'f1': round(f1_per_e[i], 4), 'support': int(s_per_e[i])} for i, label in enumerate(SENTIMENT_LABELS)},
        'confusion_matrix': cm_ens,
    },
    'timestamp': datetime.now().isoformat(),
    'status': 'completed',
}
results_list.append(ens_result)


Running EXP-04_Ensemble (LR + SVM Soft Voting)


Accuracy: 90.73%
              precision    recall  f1-score   support

    Negative       0.81      0.91      0.86       530
     Neutral       0.33      0.03      0.06        63
    Positive       0.95      0.94      0.95      1391

    accuracy                           0.91      1984
   macro avg       0.70      0.63      0.62      1984
weighted avg       0.89      0.91      0.90      1984



Ensemble model saved to ../models/EXP-04_Ensemble/


## 5. Experiment 5: IndoBERT Base

Fine-tune `indobenchmark/indobert-base-p1` with oversampled Neutral class.

In [7]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed

set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'IndoBERT Device: {DEVICE}')

MODEL_NAME = 'indobenchmark/indobert-base-p1'
NUM_LABELS = 3
MAX_LENGTH = 64
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

IndoBERT Device: cuda


In [8]:
# Oversample Neutral
neutral_mask = df['label'] == 1
if neutral_mask.sum() < 500:
    n_replicates = max(1, 500 // neutral_mask.sum())
    neutral_df = df[neutral_mask]
    neutral_oversampled = pd.concat([neutral_df] * n_replicates, ignore_index=True)
    df_bert = pd.concat([df, neutral_oversampled], ignore_index=True)
else:
    df_bert = df.copy()
print(f'After oversampling neutral: {len(df_bert)} samples')
print(f'Label dist: {df_bert["label"].value_counts().sort_index().tolist()}')

# Re-split after oversampling
X_b = np.array(df_bert['clean'].values)
y_b = np.array(df_bert['label'].values).astype(int)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.2, stratify=y_b, random_state=SEED
)
print(f'IndoBERT Train: {len(X_train_b)}, Test: {len(X_test_b)}')

After oversampling neutral: 10231 samples
Label dist: [2649, 630, 6952]
IndoBERT Train: 8184, Test: 2047


In [9]:
# Tokenizer & Dataset
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()} | {'labels': self.labels[idx]}
    def __len__(self):
        return len(self.labels)

train_ds = ReviewDataset(X_train_b, y_train_b, tokenizer_bert, MAX_LENGTH)
test_ds = ReviewDataset(X_test_b, y_test_b, tokenizer_bert, MAX_LENGTH)

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
# Model
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# Training Args
training_args = TrainingArguments(
    output_dir='../models/EXP-05_IndoBERT/checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_steps=500,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
    seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'f1_macro': f1}

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# Train
print('\nStarting IndoBERT training...')
trainer.train()
print('Training complete!')


Starting IndoBERT training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.418607,0.373466,0.880801,0.595333
2,0.259395,0.353560,0.892526,0.762010
3,0.171454,0.426589,0.900342,0.790746


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.output.LayerNor

Training complete!


In [12]:
# Evaluate
print('\nEvaluating on test set...')
eval_results = trainer.evaluate()
print(f'Test accuracy: {eval_results["eval_accuracy"]:.2%}')

preds = trainer.predict(test_ds)
y_pred_b = np.argmax(preds.predictions, axis=1)
acc_b = accuracy_score(y_test_b, y_pred_b)
p_per_b, r_per_b, f1_per_b, s_per_b = precision_recall_fscore_support(y_test_b, y_pred_b, labels=[0, 1, 2], zero_division=0)
cm_b = confusion_matrix(y_test_b, y_pred_b).tolist()
cls_b = classification_report(y_test_b, y_pred_b, target_names=SENTIMENT_LABELS, zero_division=0)

print(cls_b)

# Save artifacts
save_dir = '../models/EXP-05_IndoBERT'
os.makedirs(f'{save_dir}/checkpoints', exist_ok=True)
model_bert.save_pretrained(save_dir)
tokenizer_bert.save_pretrained(save_dir)
with open('../reports/EXP-05_IndoBERT_classification.txt', 'w') as f:
    f.write(cls_b)
print(f'Model saved to {save_dir}/')

# Store result
acc_b_val = round(acc_b, 4)
f1_b_val = round(eval_results.get('eval_f1_macro', f1_per_b.mean()), 4)
bert_result = {
    'experiment': 'EXP-05_IndoBERT',
    'model': 'indobenchmark/indobert-base-p1',
    'metrics': {
        'accuracy_test': acc_b_val,
        'f1_macro': f1_b_val,
        'per_class': {label: {'precision': round(p_per_b[i], 4), 'recall': round(r_per_b[i], 4), 'f1': round(f1_per_b[i], 4)} for i, label in enumerate(SENTIMENT_LABELS)},
        'confusion_matrix': cm_b,
    },
    'training_config': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LR, 'max_length': MAX_LENGTH, 'oversampled_neutral': True},
    'timestamp': datetime.now().isoformat(),
}
results_list.append(bert_result)


Evaluating on test set...


Test accuracy: 90.03%


              precision    recall  f1-score   support

    Negative       0.83      0.88      0.86       530
     Neutral       0.63      0.52      0.57       126
    Positive       0.95      0.94      0.95      1391

    accuracy                           0.90      2047
   macro avg       0.80      0.78      0.79      2047
weighted avg       0.90      0.90      0.90      2047



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ../models/EXP-05_IndoBERT/


## 6. Experiment 6: IndoBERT Tuned

Continue training from EXP-05 checkpoint with lower learning rate and extended max_length.

In [13]:
# Re-load data with raw split (no oversample mixing for tuned)
X_pt = np.array(df['clean'].values)
y_pt = np.array(df['label'].values).astype(int)
X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_pt, y_pt, test_size=0.2, stratify=y_pt, random_state=SEED
)
print(f'Tuned Train: {len(X_train_pt)}, Test: {len(X_test_pt)}')

# Load EXP-05 checkpoint
tuned_model = AutoModelForSequenceClassification.from_pretrained('../models/EXP-05_IndoBERT', num_labels=NUM_LABELS)
tuned_tokenizer = AutoTokenizer.from_pretrained('../models/EXP-05_IndoBERT')

train_ds_t = ReviewDataset(X_train_pt, y_train_pt, tuned_tokenizer, 96)
test_ds_t = ReviewDataset(X_test_pt, y_test_pt, tuned_tokenizer, 96)

Tuned Train: 7932, Test: 1984


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [14]:
# Tuning args (lower LR, longer max_length)
tune_args = TrainingArguments(
    output_dir='../models/EXP-06_IndoBERT_Tuned/checkpoints',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
    seed=SEED,
)

tune_trainer = Trainer(
    model=tuned_model,
    args=tune_args,
    train_dataset=train_ds_t,
    eval_dataset=test_ds_t,
    compute_metrics=compute_metrics,
)

In [15]:
# Train tuned
print('\nStarting IndoBERT tuning...')
tune_trainer.train()
print('Tuning complete!')


Starting IndoBERT tuning...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
200,0.149449,0.314454,0.926411,0.827989
400,0.118162,0.359843,0.928427,0.830237
600,0.109654,0.324867,0.932460,0.833384
800,0.143645,0.341405,0.932460,0.834408
1000,0.240194,0.333292,0.927419,0.824841
1200,0.127824,0.374689,0.919355,0.797441
1400,0.091455,0.381667,0.928931,0.811679
1600,0.109486,0.404958,0.922379,0.800084
1800,0.082697,0.419642,0.922883,0.802694


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.output.LayerNor

Tuning complete!


In [16]:
# Evaluate tuned
eval_t = tune_trainer.evaluate()
preds_t = tune_trainer.predict(test_ds_t)
y_pred_t = np.argmax(preds_t.predictions, axis=1)
acc_t = accuracy_score(y_test_pt, y_pred_t)
p_per_t, r_per_t, f1_per_t, s_per_t = precision_recall_fscore_support(y_test_pt, y_pred_t, labels=[0, 1, 2], zero_division=0)
cm_t = confusion_matrix(y_test_pt, y_pred_t).tolist()
cls_t = classification_report(y_test_pt, y_pred_t, target_names=SENTIMENT_LABELS, zero_division=0)

print(f'Test accuracy: {eval_t["eval_accuracy"]:.2%}')
print(cls_t)
print(f'Confusion Matrix:\n{np.array(cm_t)}')

# Save tuned model
tuned_dir = '../models/EXP-06_IndoBERT_Tuned'
os.makedirs(tuned_dir, exist_ok=True)
tuned_model.save_pretrained(tuned_dir)
tuned_tokenizer.save_pretrained(tuned_dir)
with open('../reports/EXP-06_IndoBERT_Tuned_classification.txt', 'w') as f:
    f.write(cls_t)
print(f'Tuned model saved to {tuned_dir}/')

tuned_result = {
    'experiment': 'EXP-06_IndoBERT_Tuned',
    'model': 'indobenchmark/indobert-base-p1 (tuned)',
    'metrics': {
        'accuracy_test': round(acc_t, 4),
        'f1_macro': round(eval_t.get('eval_f1_macro', f1_per_t.mean()), 4),
        'per_class': {label: {'precision': round(p_per_t[i], 4), 'recall': round(r_per_t[i], 4), 'f1': round(f1_per_t[i], 4)} for i, label in enumerate(SENTIMENT_LABELS)},
        'confusion_matrix': cm_t,
    },
    'training_config': {'base_model': 'EXP-05_IndoBERT', 'extra_epochs': 2, 'learning_rate': 1e-5, 'max_length': 96},
    'timestamp': datetime.now().isoformat(),
}
results_list.append(tuned_result)

Test accuracy: 93.25%
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89       530
     Neutral       0.69      0.60      0.64        63
    Positive       0.95      0.97      0.96      1391

    accuracy                           0.93      1984
   macro avg       0.85      0.82      0.83      1984
weighted avg       0.93      0.93      0.93      1984

Confusion Matrix:
[[ 468   10   52]
 [  13   38   12]
 [  40    7 1344]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tuned model saved to ../models/EXP-06_IndoBERT_Tuned/


## 7. Save Combined Results & Summary

In [17]:
# Save combined results
all_results = results_list

# Update best
best_acc = max(r['metrics'].get('accuracy', 0) or r['metrics'].get('accuracy_test', 0) for r in all_results)

# Try to merge with existing
results_path = '../reports/experiment_results.json'
if os.path.exists(results_path):
    existing = json.load(open(results_path))
    existing['experiments'] = all_results
    existing['summary'] = {
        'total': len(all_results),
        'best_accuracy': best_acc,
        'timestamp': datetime.now().isoformat(),
    }
    json.dump(existing, open(results_path, 'w'), indent=2)
else:
    summary = {
        'experiments': all_results,
        'summary': {
            'total': len(all_results),
            'best_accuracy': best_acc,
            'timestamp': datetime.now().isoformat(),
        },
    }
    json.dump(summary, open(results_path, 'w'), indent=2)

print(f'Results saved to {results_path}')

Results saved to ../reports/experiment_results.json


In [18]:
# Final summary
print(f'\n{"="*60}')
print('ALL EXPERIMENTS COMPLETE')
print('='*60)
for r in all_results:
    acc_val = r['metrics'].get('accuracy', 0) or r['metrics'].get('accuracy_test', 0)
    marker = '★ BEST' if acc_val == best_acc else ''
    print(f'  {r["experiment"]}: acc={acc_val:.2%}  {marker}')
print(f'\nBest accuracy: {best_acc:.2%}')


ALL EXPERIMENTS COMPLETE
  EXP-01_LR_TFIDF: acc=88.46%  
  EXP-02_SVM_TFIDF: acc=89.57%  
  EXP-03_LR_BoW: acc=88.00%  
  EXP-04_Ensemble: acc=90.73%  
  EXP-05_IndoBERT: acc=90.03%  
  EXP-06_IndoBERT_Tuned: acc=93.25%  ★ BEST

Best accuracy: 93.25%
